## A/B тестирование

### Сценарий эксперимента

#### Проблема  
Продукт продаёт подписки через пейволл (экран оплаты). Текущая конверсия из просмотра пейволла в покупку составляет около 10%. Команда гипотетически теряет пользователей, которые готовы платить, но считают, что базовая цена слишком высокая, или не видят ценности в коротких планах.  
#### Гипотеза  
Если добавить на экран подписки специальное предложение (скидка для годовой подписки), то конверсия в покупку вырастет с 10% до 12.5% (относительный рост +25%), при этом общий ARPU (средняя выручка на пользователя) не упадёт из-за скидки.  
#### Группы эксперимента  
Group A (Control): Стандартный экран оплаты.  
Group B (Treatment): Экран оплаты со скидкой / акцентом на годовой план.  
#### Дерево метрик теста  
**Primary metric:**  
Paywall Conversion Rate = $\frac{количество\ пользователей\ с\ событием\ subscription\_purchase}{количество\ пользователей\ с\ событием\ paywall\_view}$  
**Secondary metric:**  
ARPU = $\frac{Сумма\ completed\_orders}{Общее\ число\ пользователей\ в\ группе}$   
**Guardrail Metrics**  
D7 Retention Rate (чтобы убедиться, что новый экран не привлекает "нецелевую" аудиторию, которая удаляет приложение на следующий день)  
Failed Payment Rate = $\frac{Число\ транзакций\ со\ статусом\ failed}{Все\ транзакции}$ (проверка корректности работы оплаты)

### Дизайн эксперимента

$H_0$: разницы между группами A и B нет  
$H_1$: разница есть  
$\alpha$ (вероятность false positive) = 0.05, $\beta$ (вероятность false negative) = 0.2  
**MDE**: базовая конверсия группы A $p_1$ = 0.1, ожидаемая конверсия группы B $p_2$ = 0.125; абсолютный MDE = 2.5%, относительный MDE = 12.5% 

расчёт размера выборки, требуемой для каждой группы
$$n = \frac{\left(Z_{\alpha/2} \cdot \sqrt{2 \cdot \bar{p}(1 - \bar{p})} + Z_{\beta} \cdot \sqrt{p_1(1 - p_1) + p_2(1 - p_2)}\right)^2}{(p_2 - p_1)^2}$$

$p_1 = 0.10$ (базовая конверсия).  
$p_2 = 0.125$ (ожидаемая конверсия).  
$\bar{p} = \frac{p_1 + p_2}{2} = 0.1125$.  
$Z_{\alpha/2} \approx 1.96$ (для $\alpha = 0.05$).  
$Z_{\beta} \approx 0.84$ (для мощности $80\%$).

In [41]:
import random
import os
import numpy as np
import pandas as pd
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.proportion import proportions_ztest
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [42]:
(1.96 * np.sqrt(2 * 0.1125 * (1 - 0.1125)) + 0.84 * np.sqrt(0.1 * 0.9 + 0.125 * 0.875)) ** 2 / (0.025) ** 2

2503.7036776820173

всего в датасете 12 000 пользователей - достаточно

### Статистический анализ

#### Выгружаем данные

In [43]:
load_dotenv()

DB_USER = "postgres"
DB_PASSWORD = os.getenv("DB_PASSWORD")
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "product_analytics"

DATABASE_URL = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

np.random.seed(42)
random.seed(42)

engine = create_engine(DATABASE_URL)

In [44]:
query = """
select user_id, group_id, max(case when event_name = 'paywall_view' then 1 else 0 end) as viewed_paywall,
max(case when status = 'completed' then 1 else 0 end) as made_purchase,
sum(case when status = 'completed' then amount else 0 end) as revenue

from users
left join events
	using(user_id)
inner join ab_experiments
	using(user_id)
left join orders
	using(user_id)
where experiment_name = 'paywall_discount_v1'
group by user_id, group_id
"""

df = pd.read_sql(query, engine)
df.head()

,user_id,group_id,viewed_paywall,made_purchase,revenue
0,f1fe9931-c393-41ed-8049-260c54dc46fb,B,1,0,0.0
1,cfd99dbc-e45a-4ae6-9e49-7b2365545ee5,A,1,0,0.0
2,ac646ce9-86a5-4d15-8533-5ddb9b4454f9,A,1,0,0.0
3,2adc8b10-0cc3-45d0-82cd-9c590491064f,B,1,0,0.0
4,390ce12f-5880-4d4c-a17b-bd74e0f201e4,A,0,0,0.0


In [45]:
df = df[df['viewed_paywall'] == 1]

#### Проверка Sample Ratio Mismatch

используем критерий Хи-квадрат о соответствии

In [60]:
observed = df['group_id'].value_counts()
expected = [len(df) / 2, len(df) / 2]

chi2_srm, p_val_srm = stats.chisquare(f_obs=observed, f_exp=expected)

print(observed, '\n pvalue =', round(p_val_srm.item(), 3))
if p_val_srm < 0.01:
    print("SRM detected")
else:
    print("No SRM detected")

group_id
B    4107
A    4034
Name: count, dtype: int64 
 pvalue = 0.418
No SRM detected


#### Анализ конверсии

In [47]:
conversions = df.groupby('group_id')['made_purchase'].agg(['sum', 'count', 'mean'])
conversions['conversion_rate_%'] = conversions['mean'] * 100
print(conversions)

          sum  count      mean  conversion_rate_%
group_id                                         
A         392   4034  0.097174           9.717402
B         491   4107  0.119552          11.955198


In [ ]:
count_a = conversions.loc['A', 'sum']
count_b = conversions.loc['B', 'sum']
nobs_a = conversions.loc['A', 'count']
nobs_b = conversions.loc['B', 'count']

p_a = conversions.loc['A', 'mean']
p_b = conversions.loc['B', 'mean']

# Z-статистика и p-value
z_stat, p_val_z = proportions_ztest([count_b, count_a], [nobs_b, nobs_a])

# расчёт доверительного интервала
p_diff = p_b - p_a # центральная точка интервала
se_diff = np.sqrt((p_a * (1 - p_a) / nobs_a) + (p_b * (1 - p_b) / nobs_b)) # стандартное отклонение
z_critical = stats.norm.ppf(0.975)

ci_diff_lower = p_diff - z_critical * se_diff
ci_diff_upper = p_diff + z_critical * se_diff

print("=== Z-test & Confidence Interval ===")
print(f"CR Group A: {p_a*100:.2f}% | CR Group B: {p_b*100:.2f}%")
print(f"Absolute Lift (p_B - p_A): {p_diff*100:+.2f}%")
print(f"95% CI for Difference: [{ci_diff_lower*100:+.2f}%, {ci_diff_upper*100:+.2f}%]")
print(f"Z-statistic: {z_stat:.4f}, p-value: {p_val_z:.4e}\n")

=== 1. Z-test & Confidence Interval ===
CR Group A: 9.72% | CR Group B: 11.96%
Absolute Lift (p_B - p_A): +2.24%
95% CI for Difference: [+0.89%, +3.59%]
Z-statistic: 3.2464, p-value: 1.1688e-03



Для таблицы сопряжённости 2 * 2 двухсторонний двухвыборочный Z-тест и критерий Хи-квадрат математически идентичны. Посмотрим, так ли это на нашем датасете:

In [50]:
# критерий хи-квадрат
contingency_table = [
    [count_a, nobs_a - count_a],
    [count_b, nobs_b - count_b]
]

chi2_stat, p_val_chi2, dof, expected = stats.chi2_contingency(contingency_table, correction=False)

print("=== 2. Chi-Square Test ===")
print(f"Contingency Table:\n{np.array(contingency_table)}")
print(f"Chi2-statistic: {chi2_stat:.4f}, p-value: {p_val_chi2:.4e}")

=== 2. Chi-Square Test ===
Contingency Table:
[[ 392 3642]
 [ 491 3616]]
Chi2-statistic: 10.5391, p-value: 1.1688e-03


#### Анализ ARPU через Bootstrap

У метрики сильная скошенность - большая часть пользователей ничего не платит, оставшаяся часть платит фиксированные суммы. Кроме этого, у нас большое число выбросов - некоторые пользователи могут платить много и из-за этого сдвигать среднее значение. Т-критерий Стьюдента может дать искаженную оценку, поэтому используем Bootstrap.

In [55]:
def bootstrap_arpu_diff(group_a, group_b, n_iterations=5000, ci=95):
    diffs = []
    n_a, n_b = len(group_a), len(group_b)
    
    np.random.seed(67)
    for _ in range(n_iterations):
        sample_a = np.random.choice(group_a, size=n_a, replace=True)
        sample_b = np.random.choice(group_b, size=n_b, replace=True)
        diffs.append(np.mean(sample_b) - np.mean(sample_a))
        
    lower_p = (100 - ci) / 2
    upper_p = 100 - lower_p
    ci_lower = np.percentile(diffs, lower_p)
    ci_upper = np.percentile(diffs, upper_p)
    
    return np.mean(diffs), (ci_lower, ci_upper)

rev_a = df[df['group_id'] == 'A']['revenue'].values
rev_b = df[df['group_id'] == 'B']['revenue'].values

mean_diff, (ci_low, ci_high) = bootstrap_arpu_diff(rev_a, rev_b)

print("\n--- ARPU Bootstrap Analysis ---")
print(f"Mean ARPU Group A: ${np.mean(rev_a):.2f}")
print(f"Mean ARPU Group B: ${np.mean(rev_b):.2f}")
print(f"ARPU Lift (B - A): ${mean_diff:.2f}")
print(f"95% Confidence Interval for ARPU Lift: [${ci_low:.2f}, ${ci_high:.2f}]")


--- ARPU Bootstrap Analysis ---
Mean ARPU Group A: $12.62
Mean ARPU Group B: $17.17
ARPU Lift (B - A): $4.56
95% Confidence Interval for ARPU Lift: [$2.10, $7.02]
